<a href="https://colab.research.google.com/github/RojasG4mer/Fundamentos-de-Procesamiento-Digital-de-Imagenes/blob/main/Usando_ASSCI_como_pixeles_de_intensidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# De intensidades a ASCCI

Crear un código que transforme las intensidades de una imagen a letras del código ASCCI.

## Bibliotecas

In [ ]:
!pip install fpdf

In [ ]:
from google.colab import drive
from PIL import Image
import io


## Montar el drive
Esto para poder leer el archivo desde el drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Código

### División en bloques de la imagen para imprimir en cada hoja

In [ ]:
def procesar_por_bloques(img, ancho_salida, alto_salida):
    # Dimensiones originales
    ancho, largo = img.size

    # Calcular el tamaño del bloque (en píxeles)
    ancho_bloque = ancho_salida
    alto_bloque = alto_salida

    print(f"Dimensiones imagen: {ancho}x{largo}")
    print(f"Tamaño de cada bloque: {ancho_bloque}x{alto_bloque}")

    bloques = []

    # Recorrer la imagen como una cuadrícula:
    for y in range(0, largo, alto_bloque):
        # 'x' controla la posición horizontal (columnas)
        for x in range(0, ancho, ancho_bloque):

            # Definimos las coordenadas del rectángulo de corte
            # (izquierda, arriba, derecha, abajo)
            izquierda = x
            arriba = y
            derecha = x + ancho_bloque
            abajo = y + alto_bloque

            # Si el último bloque se sale de la imagen, limitamos 'derecha' y 'abajo'
            # al borde real de la imagen para evitar errores o áreas negras vacías.
            if derecha > ancho:
                derecha = ancho
            if abajo > largo:
                abajo = largo

            caja = (izquierda, arriba, derecha, abajo)

            # 4. Recortar el bloque
            bloque = img.crop(caja)

            # Aquí puedes guardar el bloque, procesarlo a ASCII, etc.
            bloques.append(bloque)

            # --- Visualización de progreso (Opcional) ---
            print(f"Bloque procesado: Coordenadas {caja} - Tamaño real: {bloque.size}")

    return bloques

In [ ]:
def imagen_a_ascii_por_bloques(ruta_imagen, ancho_salida=80, alto_salida=120):
    try:
        # Cargar imagen y convertir a grises
        img = Image.open(ruta_imagen).convert('L')

        # Redimensionar
        # Valores para hacer la iamgen más chiquita y ver lo importante:
        ancho_preferente = 200
        alto_preferente = 100
        img = img.resize((ancho_preferente, alto_preferente), Image.Resampling.LANCZOS)

        # Convertir a escala de grises
        img = img.convert('L')

        # Dividir en bloques
        bloques = procesar_por_bloques(img, ancho_salida, alto_salida)

        print(f"Procesando {len(bloques)} bloques (páginas)...")

        # Caracteres ASCII (Ordenados de oscuro a claro)
        chars = "@%N#QOPnopq^*+=-:. "

        paginas_ascii = [] # Aquí guardaremos cada bloque convertido

        # Iteramos sobre CADA BLOQUE individualmente
        for i, bloque in enumerate(bloques):

            pixels = bloque.getdata()
            ascii_str = ""

            div = 256 / len(chars)

            for pixel in pixels:
                ascii_str += chars[int(pixel // div)]

            # Formateo (cortar las líneas según el ancho de ESTE bloque)
            ancho_actual = bloque.width
            ascii_bloque = ""

            for j in range(0, len(ascii_str), ancho_actual):
                ascii_bloque += ascii_str[j:j + ancho_actual] + "\n"

            # Agregamos el resultado a la lista
            paginas_ascii.append(ascii_bloque)
            # print(f"Bloque {i+1} convertido a ASCII.")

        return paginas_ascii

    except FileNotFoundError:
        return ["Error: No se encontró el archivo."]
    except Exception as e:
        return [f"Error inesperado: {e}"]

## Lectura de la foto en drive

In [ ]:
# Para recordar:
# En Colab, la ruta suele empezar con '/content/drive/MyDrive/...'
# Si está en una carpeta específica se debe ponerla.

nombre_imagen = "Popi_fachero.jpg"
carpeta_en_drive = "Usando ASSCI como pixeles" # <- CAMBIA ESTO (o déjalo vacío "" si está en la raíz)
# Ruta general para la carpeta dentro de procesamiento de imagenes
ruta_general = "/content/drive/MyDrive/Procesamiento Digital del Imágenes/"


# Corroboramos que si esté en la carpeta la imagen para evitar errores:
if carpeta_en_drive:
    ruta_completa = f"{ruta_general}{carpeta_en_drive}/{nombre_imagen}"
else:
    ruta_completa = f"/content/drive/MyDrive/{nombre_imagen}"

print(f"Buscando imagen en: {ruta_completa}")

# Ejecutar conversión
resultado = imagen_a_ascii_por_bloques(ruta_completa, 200, 100)
# print(resultado)

Buscando imagen en: /content/drive/MyDrive/Procesamiento Digital del Imágenes/Usando ASSCI como pixeles/Popi_fachero.jpg
Dimensiones imagen: 200x100
Tamaño de cada bloque: 200x100
Bloque procesado: Coordenadas (0, 0, 200, 100) - Tamaño real: (200, 100)
Procesando 1 bloques (páginas)...
[':---:-::::::::::::--:---------===-==---============pOOOOQQOOOOOOnnOPOOPPPPPPPPnnQOPPPPPPOPPPnPOOOPPoOOnOOOOPOOOQQQOOOOOOQQOQOOOQOOOOOPP#OnPPnnnnnnnnnoonnoooooooonnnnooopponnopopoopppppqqppppppppppppqq\n:--:---::::::--:--:-:--=------==--=-==-====+========pOPOOQ#QOOPOnoPnOOPPPPnnPnPnPOOOPPPPPnPOnPQQPOPOQQnOQOPOOOQO#QOOOOOOOQOQOOQOOOOOPPQPnPPnnnnnnnnnooooooooooooonnnopppppnooopopopooppppqppqqpppppppppp\n::::::::::::::::----:----------=-=-=-=======+========pPnPPO#OPOOPnnPOOnnPOPPPnnPnnPPPPPOOnPnoPOOPPPQ#OPPQOPOPOQOQQOOOOPPOQOOOOQQOQQOnPOPPPnnnnnnnnoooonnooooooopononoppoooooooopoppopppppqqppqppppppppop\n::::::::::::::::::::::::------------==-===============pPPPPOOPPOnPOPPPnPPPPnPPnPPnnPPPPOPPPnoPOOOPPQQOOO

# Haciendo el PDF

In [ ]:
from fpdf import FPDF

In [ ]:
def guardar_multipage_pdf(lista_textos, ruta_salida):
    pdf = FPDF(orientation='P', unit='mm', format='A4')

    # Configuración de fuente (Usa Courier para alinear)
    pdf.set_font("Courier", size=6)
    altura_linea = 2.5

    for i, texto_pagina in enumerate(lista_textos):
        pdf.add_page() # Nueva página por cada bloque

        # Opcional: Escribir número de bloque
        # pdf.cell(0, 5, txt=f"Bloque {i+1}", ln=1, align='L')

        for linea in texto_pagina.split('\n'):
            pdf.cell(0, altura_linea, txt=linea, ln=1, align='C')

    pdf.output(ruta_salida)
    print(f"PDF guardado en: {ruta_salida}")

In [ ]:
# Usamos la misma carpeta para guardar el pdf:
# ruta_general
nombre_pdf = "popi_fachero_enASCII.pdf"

if carpeta_en_drive:
    ruta_pdf = f"{ruta_general}{carpeta_en_drive}/{nombre_pdf}"
else:
    ruta_pdf = f"{ruta_general}{carpeta_en_drive}/{nombre_pdf}"

# Generar
print(f"Generando PDF en: {ruta_pdf}...")
guardar_multipage_pdf(resultado, ruta_pdf)

Generando PDF en: /content/drive/MyDrive/Procesamiento Digital del Imágenes/Usando ASSCI como pixeles/popi_fachero_enASCII.pdf...
PDF guardado en: /content/drive/MyDrive/Procesamiento Digital del Imágenes/Usando ASSCI como pixeles/popi_fachero_enASCII.pdf
